# Gerador de Textos Especializado: Prompting vs Ajuste Fino com LoRA

**Projeto Final — IA Generativa (Tema 02)**

**Objetivo:** gerar **resumos de notícias de Ciência e Tecnologia** em português e comparar duas abordagens aplicadas ao **mesmo modelo pequeno**:

| Abordagem | Descrição |
|---|---|
| **Prompting zero-shot** | Modelo pré-treinado recebe apenas a instrução e a notícia |
| **Prompting one-shot** | Modelo pré-treinado recebe a instrução, **um exemplo resolvido** e a notícia |
| **Ajuste fino (QLoRA)** | Mesmo modelo, ajustado com adaptadores LoRA em pares notícia → resumo |

**Avaliação:** em um conjunto de teste separado, comparamos **fidelidade** (ROUGE e BERTScore em relação ao resumo humano), **clareza** (avaliação manual) e **tempo de geração**.

**Base teórica:** *Generative AI with Python and PyTorch*, 2ª ed., caps. 3 a 10 (o notebook segue a estrutura do notebook de *Instruction Tuning* do cap. 9 apresentado em aula).

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ❗ <b>Este notebook requer GPU.</b> No Google Colab: <i>Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4</i>.</p>

## Relembrando: LoRA e QLoRA

No ajuste fino convencional, toda a matriz de pesos $W_0$ é atualizada. No **LoRA**, $W_0$ fica **congelada** e a atualização é aproximada por duas matrizes de baixo posto:

$$
W = W_0 + \frac{\alpha}{r} W_a W_b
$$

em que $W_a \in \mathbb{R}^{d \times r}$, $W_b \in \mathbb{R}^{r \times k}$ e $r \ll \min(d, k)$. Assim, treinamos menos de 1% dos parâmetros.

No **QLoRA**, o modelo base congelado é carregado em **4 bits (NF4)**, reduzindo o consumo de memória e permitindo treinar na GPU T4 (16 GB) do Colab.

## 1. Preparação do ambiente

In [ ]:
!pip install -q -U "huggingface_hub>=0.34,<1.0" "datasets<4.0.0"
!pip install -q -U accelerate peft bitsandbytes transformers trl
!pip install -q rouge_score bert_score

In [ ]:
# ============================================================
# IMPORTAÇÃO DAS BIBLIOTECAS
# ============================================================

import os
import re
import time
import random

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from IPython.display import display, Markdown

print(
    f"GPU disponível: {torch.cuda.is_available()} | "
    f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Nenhuma'}"
)

In [ ]:
# ============================================================
# REPRODUTIBILIDADE
# ============================================================

# Fixamos as sementes para que a seleção dos exemplos e o
# treinamento possam ser reproduzidos.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 2. Configurações do experimento

Todas as escolhas importantes ficam nesta célula, para facilitar a repetição dos experimentos.

**Sobre o modelo:** usamos o **Qwen2.5-1.5B-Instruct**, que tem bom suporte a português e **não exige pedido de acesso** no Hugging Face. O **Llama 3.2-1B-Instruct** (usado em aula) também funciona: basta trocar `MODELO_BASE`, fazer login com seu token e ter o acesso aprovado pela Meta.

In [ ]:
# ============================================================
# CONFIGURAÇÃO DO MODELO BASE
# ============================================================

QWEN_1_5B  = "Qwen/Qwen2.5-1.5B-Instruct"       # padrão (sem pedido de acesso)
QWEN_0_5B  = "Qwen/Qwen2.5-0.5B-Instruct"       # alternativa mais rápida
LLAMA_3_2  = "meta-llama/Llama-3.2-1B-Instruct" # modelo usado em aula (requer acesso)

MODELO_BASE = QWEN_1_5B

# Nome do diretório onde o adaptador LoRA será salvo.
NOVO_MODELO = MODELO_BASE.split("/")[-1] + "-resumo-CT-LoRA"

# ============================================================
# CONFIGURAÇÃO DOS DADOS
# ============================================================

DATASET_NOME = "recogna-nlp/recognasumm"

# Quantidade de exemplos para treino e teste.
N_TREINO = 1000   # aumente para 2000–3000 se houver tempo de GPU
N_TESTE  = 50     # exemplos usados na avaliação comparativa

# As notícias são truncadas para caber no contexto e reduzir o
# tempo de treino. Registre esta escolha como limitação.
MAX_PALAVRAS_NOTICIA = 350

# ============================================================
# CONFIGURAÇÃO DA GERAÇÃO
# ============================================================

MAX_NEW_TOKENS = 120   # tamanho máximo do resumo gerado
USAR_ONE_SHOT  = True  # inclui a abordagem de prompting one-shot

print("Modelo base:", MODELO_BASE)
print("Adaptador será salvo em:", NOVO_MODELO)

In [ ]:
# ============================================================
# LOGIN NO HUGGING FACE (somente se usar o Llama)
# ============================================================

# Nunca escreva o token diretamente no notebook!
# No Colab, cadastre-o em "Secrets" (ícone de chave) com o
# nome HF_TOKEN e ative o acesso para este notebook.

if MODELO_BASE == LLAMA_3_2:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)

## 3. Dados: RecognaSumm

O **RecognaSumm** (Paiola et al., PROPOR 2024) é um dataset de sumarização em português do Brasil com cerca de 135 mil notícias coletadas de agências e portais, cada uma acompanhada de um sumário. É distribuído sob licença **MIT** no Hugging Face (`recogna-nlp/recognasumm`) e já vem dividido em `train`, `validation` e `test`.

**Domínio escolhido:** filtramos apenas as notícias da categoria **Ciência e Tecnologia**.

Colunas principais: `Titulo`, `Subtitulo`, `Noticia`, `Categoria`, `Sumario`.

In [ ]:
# ============================================================
# CARREGAMENTO DO DATASET
# ============================================================

dataset = load_dataset(DATASET_NOME, cache_dir="./cache")
print(dataset)

In [ ]:
# ============================================================
# EXPLORAÇÃO: CATEGORIAS DISPONÍVEIS
# ============================================================

# Conferimos o nome exato das categorias antes de filtrar.
print(pd.Series(dataset["train"]["Categoria"]).value_counts())

In [ ]:
# ============================================================
# FILTRO DO DOMÍNIO: CIÊNCIA E TECNOLOGIA
# ============================================================

def eh_ciencia_tecnologia(exemplo):
    cat = (exemplo["Categoria"] or "").lower()
    return ("ciência" in cat) or ("tecnologia" in cat) or ("science" in cat)

def eh_valido(exemplo):
    # Remove exemplos com notícia ou sumário vazios/muito curtos.
    noticia = exemplo["Noticia"] or ""
    sumario = exemplo["Sumario"] or ""
    return len(noticia.split()) >= 80 and len(sumario.split()) >= 10

treino_ct = (dataset["train"]
             .filter(eh_ciencia_tecnologia)
             .filter(eh_valido)
             .shuffle(seed=SEED))

teste_ct = (dataset["test"]
            .filter(eh_ciencia_tecnologia)
            .filter(eh_valido)
            .shuffle(seed=SEED))

print("Treino (Ciência e Tecnologia):", len(treino_ct))
print("Teste  (Ciência e Tecnologia):", len(teste_ct))

# Seleciona os subconjuntos usados no experimento.
treino_ct = treino_ct.select(range(min(N_TREINO, len(treino_ct))))
teste_ct  = teste_ct.select(range(min(N_TESTE,  len(teste_ct))))

print(f"\nUsados no experimento -> treino: {len(treino_ct)} | teste: {len(teste_ct)}")

In [ ]:
# ============================================================
# ESTATÍSTICAS DE TAMANHO (para o relatório)
# ============================================================

def stats(textos):
    tam = [len(t.split()) for t in textos]
    return f"média {np.mean(tam):.0f} | mediana {np.median(tam):.0f} | máx {np.max(tam)} palavras"

print("Notícias:", stats(treino_ct["Noticia"]))
print("Sumários:", stats(treino_ct["Sumario"]))

In [ ]:
# ============================================================
# VISUALIZAÇÃO DE UM EXEMPLO
# ============================================================

ex = treino_ct[0]
display(Markdown(f"**Título:** {ex['Titulo']}"))
display(Markdown(f"**Notícia (início):** {' '.join(ex['Noticia'].split()[:120])} ..."))
display(Markdown(f"**Sumário de referência:** {ex['Sumario']}"))

## 4. Tokenizador e formato das instruções

Assim como no notebook de aula, cada exemplo vira um texto com **instrução do sistema**, **entrada do usuário** e **resposta esperada**.

Em vez de escrever os marcadores especiais à mão (`<|start_header_id|>`, `<|im_start|>`...), usamos `tokenizer.apply_chat_template`, que aplica automaticamente o formato correto de cada modelo (Qwen ou Llama).

In [ ]:
# ============================================================
# CARREGAMENTO DO TOKENIZADOR
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE, cache_dir="./cache")

# Alguns modelos (ex.: Llama) não definem token de padding.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("EOS:", tokenizer.eos_token, "| PAD:", tokenizer.pad_token)

In [ ]:
# ============================================================
# MONTAGEM DAS MENSAGENS (INSTRUÇÃO + NOTÍCIA + RESUMO)
# ============================================================

INSTRUCAO_SISTEMA = (
    "Você é um assistente especializado em jornalismo de ciência e tecnologia. "
    "Resuma a notícia em português, em no máximo 3 frases, de forma objetiva e "
    "fiel ao texto, sem inventar informações."
)

def truncar(texto, max_palavras=MAX_PALAVRAS_NOTICIA):
    palavras = texto.split()
    return " ".join(palavras[:max_palavras])

def montar_mensagens(noticia, resumo=None, exemplo_one_shot=None):
    msgs = [{"role": "system", "content": INSTRUCAO_SISTEMA}]

    # Prompting one-shot: um exemplo resolvido antes da notícia alvo.
    if exemplo_one_shot is not None:
        msgs.append({"role": "user",
                     "content": f"Notícia:\n{truncar(exemplo_one_shot['Noticia'])}"})
        msgs.append({"role": "assistant", "content": exemplo_one_shot["Sumario"]})

    msgs.append({"role": "user", "content": f"Notícia:\n{truncar(noticia)}"})

    # No treino, a resposta esperada é incluída.
    if resumo is not None:
        msgs.append({"role": "assistant", "content": resumo})
    return msgs

def texto_treino(noticia, resumo):
    # Exemplo completo (instrução + notícia + resumo) para o SFT.
    return tokenizer.apply_chat_template(
        montar_mensagens(noticia, resumo), tokenize=False)

def prompt_inferencia(noticia, exemplo_one_shot=None):
    # Apenas instrução + notícia; o modelo completa com o resumo.
    return tokenizer.apply_chat_template(
        montar_mensagens(noticia, exemplo_one_shot=exemplo_one_shot),
        tokenize=False, add_generation_prompt=True)

print(texto_treino(treino_ct[0]["Noticia"], treino_ct[0]["Sumario"])[:1500])

In [ ]:
# ============================================================
# DATASET FORMATADO PARA O SFTTrainer
# ============================================================

dataset_instrucoes = Dataset.from_list([
    {"text": texto_treino(ex["Noticia"], ex["Sumario"])}
    for ex in treino_ct
])

# Tamanho em tokens (confere se cabe no limite de sequência).
n_tokens = [len(tokenizer(t)["input_ids"]) for t in dataset_instrucoes["text"]]
print(dataset_instrucoes)
print(f"Tokens por exemplo: média {np.mean(n_tokens):.0f} | máx {np.max(n_tokens)}")

## 5. Configurações de quantização e LoRA

In [ ]:
# ============================================================
# QUANTIZAÇÃO EM 4 BITS (QLoRA)
# ============================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # pesos do modelo base em 4 bits
    bnb_4bit_quant_type="nf4",             # NormalFloat 4, recomendado para QLoRA
    bnb_4bit_compute_dtype=torch.float16,  # T4 não tem suporte nativo a BF16
    bnb_4bit_use_double_quant=True,        # quantiza também as constantes
)

In [ ]:
# ============================================================
# HIPERPARÂMETROS DO LoRA
# ============================================================

# Diferenças em relação ao notebook de aula (r=8, alpha=8, só q/v):
# - Resumir exige reescrever o texto, uma mudança de comportamento
#   maior que gerar SQL curto. Por isso usamos r=16 e aplicamos
#   o LoRA também em k_proj e o_proj.
# - alpha = 2*r é uma escolha comum (escala alpha/r = 2).

lora_r       = 16
lora_alpha   = 32
lora_dropout = 0.05

peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

print(f"Fator de escala LoRA (alpha/r): {lora_alpha / lora_r:.2f}")

## 6. Carregamento do modelo base e aplicação do LoRA

In [ ]:
# ============================================================
# MODELO BASE QUANTIZADO
# ============================================================

model = AutoModelForCausalLM.from_pretrained(
    MODELO_BASE,
    quantization_config=bnb_config,
    device_map={"": 0},
    dtype=torch.float16,
    cache_dir="./cache",
)

# Prepara o modelo quantizado para treino (habilita gradientes
# na entrada e estabiliza camadas de normalização).
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# Aplica os adaptadores LoRA.
peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

## 7. Treinamento (Supervised Fine-Tuning)

In [ ]:
# ============================================================
# PARÂMETROS DE TREINAMENTO
# ============================================================

training_arguments = SFTConfig(
    output_dir=NOVO_MODELO,
    num_train_epochs=1,
    fp16=True,
    bf16=False,
    loss_type="nll",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,   # batch efetivo = 4
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    seed=SEED,
)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=dataset_instrucoes,
    processing_class=tokenizer,
    args=training_arguments,
)

# Mesmo ajuste do notebook de aula: parâmetros treináveis em BF16
# são convertidos para FP32 (a T4 não suporta BF16 nativamente).
for name, param in trainer.model.named_parameters():
    if param.requires_grad and param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float32)

print("Exemplos de treino:", len(trainer.train_dataset))

In [ ]:
# ============================================================
# TREINO
# ============================================================

resultado_treino = trainer.train()

tempo_treino_min = resultado_treino.metrics["train_runtime"] / 60
print(f"\nTempo de treinamento: {tempo_treino_min:.1f} min")
print(f"Loss final de treino: {resultado_treino.metrics['train_loss']:.4f}")

In [ ]:
# ============================================================
# CURVA DE PERDA (para o relatório)
# ============================================================

logs = [l for l in trainer.state.log_history if "loss" in l]
plt.figure(figsize=(7, 3.5))
plt.plot([l["step"] for l in logs], [l["loss"] for l in logs], marker="o")
plt.xlabel("Passo de treinamento")
plt.ylabel("Loss (NLL)")
plt.title("Curva de perda do ajuste fino com LoRA")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SALVAR O ADAPTADOR LoRA
# ============================================================

# Salva apenas os pesos do LoRA (poucos MB), não o modelo inteiro.
trainer.model.save_pretrained(NOVO_MODELO)
tokenizer.save_pretrained(NOVO_MODELO)
!du -sh {NOVO_MODELO}

# Opcional: compactar e baixar para não perder ao fechar o Colab.
# !zip -r {NOVO_MODELO}.zip {NOVO_MODELO}
# from google.colab import files; files.download(f"{NOVO_MODELO}.zip")

## 8. Geração com as três abordagens

Usamos **o mesmo modelo carregado** para as três abordagens:

- com `peft_model.disable_adapter()`, o LoRA é desligado e temos o **modelo base original** (prompting);
- sem esse contexto, o LoRA fica ativo e temos o **modelo ajustado**.

A geração é **determinística** (`do_sample=False`) para que a comparação seja justa, e medimos o tempo de cada geração.

In [ ]:
# ============================================================
# FUNÇÃO DE GERAÇÃO COM MEDIÇÃO DE TEMPO
# ============================================================

peft_model.eval()
peft_model.config.use_cache = True

# Exemplo fixo para o prompting one-shot (retirado do TREINO,
# nunca do teste).
EXEMPLO_ONE_SHOT = treino_ct[1]

@torch.no_grad()
def gerar(prompt):
    inputs = tokenizer(prompt, return_tensors="pt",
                       add_special_tokens=False).to(peft_model.device)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = peft_model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
    )
    torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    # Decodifica apenas os tokens novos (o resumo).
    novos = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(novos, skip_special_tokens=True).strip(), dt

def resumir(noticia, abordagem):
    if abordagem == "lora":
        return gerar(prompt_inferencia(noticia))
    one_shot = EXEMPLO_ONE_SHOT if abordagem == "one_shot" else None
    with peft_model.disable_adapter():
        return gerar(prompt_inferencia(noticia, exemplo_one_shot=one_shot))

ABORDAGENS = ["zero_shot"] + (["one_shot"] if USAR_ONE_SHOT else []) + ["lora"]
print("Abordagens avaliadas:", ABORDAGENS)

In [ ]:
# ============================================================
# TESTE RÁPIDO EM UM EXEMPLO
# ============================================================

ex = teste_ct[0]
display(Markdown(f"**Referência (humana):** {ex['Sumario']}"))
for ab in ABORDAGENS:
    texto, dt = resumir(ex["Noticia"], ab)
    display(Markdown(f"**{ab}** ({dt:.1f}s): {texto}"))

In [ ]:
# ============================================================
# GERAÇÃO NO CONJUNTO DE TESTE
# ============================================================

# Pode levar de 15 a 30 minutos na T4, dependendo de N_TESTE.
from tqdm.auto import tqdm

resultados = []
for i, ex in enumerate(tqdm(teste_ct, desc="Gerando resumos")):
    linha = {"id": i, "titulo": ex["Titulo"], "referencia": ex["Sumario"]}
    for ab in ABORDAGENS:
        texto, dt = resumir(ex["Noticia"], ab)
        linha[f"resumo_{ab}"] = texto
        linha[f"tempo_{ab}"]  = dt
    resultados.append(linha)

df = pd.DataFrame(resultados)
df.to_csv("resultados_brutos.csv", index=False)
df.head()

## 9. Avaliação

### 9.1 Fidelidade: ROUGE e BERTScore

- **ROUGE-1 / ROUGE-2 / ROUGE-L:** sobreposição de palavras, pares de palavras e maior subsequência comum com o resumo humano.
- **BERTScore (F1):** similaridade semântica usando embeddings de um BERT multilíngue; captura paráfrases que o ROUGE não pega.

⚠️ O tokenizador padrão do `rouge_score` remove caracteres fora de `a-z0-9`, o que **quebra palavras acentuadas** ("ciência" → "ci", "ncia"). Por isso usamos um tokenizador próprio que mantém os acentos.

In [ ]:
# ============================================================
# ROUGE COM TOKENIZADOR ADEQUADO AO PORTUGUÊS
# ============================================================

from rouge_score import rouge_scorer

class TokenizadorPT:
    def tokenize(self, texto):
        return re.findall(r"\w+", texto.lower())

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"],
                                  tokenizer=TokenizadorPT())

for ab in ABORDAGENS:
    for m in ["rouge1", "rouge2", "rougeL"]:
        df[f"{m}_{ab}"] = [
            scorer.score(ref, hyp)[m].fmeasure
            for ref, hyp in zip(df["referencia"], df[f"resumo_{ab}"])
        ]

In [ ]:
# ============================================================
# BERTScore
# ============================================================

from bert_score import score as bert_score

for ab in ABORDAGENS:
    _, _, F1 = bert_score(df[f"resumo_{ab}"].tolist(), df["referencia"].tolist(),
                          lang="pt", verbose=False)
    df[f"bertscore_{ab}"] = F1.numpy()

In [ ]:
# ============================================================
# TABELA COMPARATIVA
# ============================================================

tabela = pd.DataFrame({
    ab: {
        "ROUGE-1":          df[f"rouge1_{ab}"].mean(),
        "ROUGE-2":          df[f"rouge2_{ab}"].mean(),
        "ROUGE-L":          df[f"rougeL_{ab}"].mean(),
        "BERTScore F1":     df[f"bertscore_{ab}"].mean(),
        "Tempo médio (s)":  df[f"tempo_{ab}"].mean(),
        "Palavras (média)": df[f"resumo_{ab}"].str.split().str.len().mean(),
    } for ab in ABORDAGENS
}).T

tabela.loc["referência humana", "Palavras (média)"] = df["referencia"].str.split().str.len().mean()
display(tabela.round(3))
print(f"Tempo de treinamento do LoRA: {tempo_treino_min:.1f} min (custo único)")
tabela.round(4).to_csv("tabela_comparativa.csv")

In [ ]:
# ============================================================
# GRÁFICOS COMPARATIVOS
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

metricas = ["ROUGE-1", "ROUGE-L", "BERTScore F1"]
t = tabela.loc[ABORDAGENS, metricas]
x = np.arange(len(metricas))
w = 0.8 / len(ABORDAGENS)
for k, ab in enumerate(ABORDAGENS):
    axes[0].bar(x + k * w, t.loc[ab].values, w, label=ab)
axes[0].set_xticks(x + w * (len(ABORDAGENS) - 1) / 2)
axes[0].set_xticklabels(metricas)
axes[0].set_title("Fidelidade em relação ao resumo humano")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(ABORDAGENS, tabela.loc[ABORDAGENS, "Tempo médio (s)"])
axes[1].set_title("Tempo médio de geração por resumo (s)")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("comparacao.png", dpi=150)
plt.show()

### 9.2 Clareza e erros de fidelidade: avaliação manual

Métricas automáticas não medem bem **clareza** nem detectam **informações inventadas**. Exportamos uma planilha com os resumos lado a lado para avaliação manual de uma amostra (sugestão: 15 notícias).

Critérios sugeridos (nota de 1 a 5):
- **Clareza:** o texto é fluente, objetivo e fácil de entender?
- **Fidelidade:** há fatos, números ou nomes inventados ou trocados? Anote o erro na coluna de observações.

Depois de preencher, envie o arquivo de volta ao Colab e rode a célula seguinte.

In [ ]:
# ============================================================
# EXPORTAÇÃO DA PLANILHA DE AVALIAÇÃO MANUAL
# ============================================================

N_MANUAL = 15
cols = ["id", "titulo", "referencia"] + [f"resumo_{ab}" for ab in ABORDAGENS]
manual = df[cols].head(N_MANUAL).copy()
for ab in ABORDAGENS:
    manual[f"clareza_{ab}"]    = ""   # nota de 1 a 5
    manual[f"fidelidade_{ab}"] = ""   # nota de 1 a 5
manual["observacoes"] = ""

manual.to_csv("avaliacao_manual.csv", index=False, encoding="utf-8-sig")
print("Planilha salva: avaliacao_manual.csv (abra no Excel ou Google Sheets)")
# from google.colab import files; files.download("avaliacao_manual.csv")

In [ ]:
# ============================================================
# RESULTADO DA AVALIAÇÃO MANUAL (após preencher a planilha)
# ============================================================

if os.path.exists("avaliacao_manual_preenchida.csv"):
    av = pd.read_csv("avaliacao_manual_preenchida.csv")
    notas = pd.DataFrame({
        ab: {
            "Clareza (1–5)":    pd.to_numeric(av[f"clareza_{ab}"], errors="coerce").mean(),
            "Fidelidade (1–5)": pd.to_numeric(av[f"fidelidade_{ab}"], errors="coerce").mean(),
        } for ab in ABORDAGENS
    }).T
    display(notas.round(2))
else:
    print("Envie 'avaliacao_manual_preenchida.csv' para ver as médias.")

## 10. Exemplos lado a lado

Exemplos qualitativos para a apresentação: o melhor e o pior caso do modelo ajustado, segundo o ROUGE-L.

In [ ]:
# ============================================================
# MELHOR E PIOR CASO DO MODELO AJUSTADO
# ============================================================

def mostrar(i):
    linha = df.iloc[i]
    display(Markdown(f"### {linha['titulo']}"))
    display(Markdown(f"**Referência:** {linha['referencia']}"))
    for ab in ABORDAGENS:
        display(Markdown(
            f"**{ab}** (ROUGE-L {linha[f'rougeL_{ab}']:.2f}, "
            f"{linha[f'tempo_{ab}']:.1f}s): {linha[f'resumo_{ab}']}"))
    display(Markdown("---"))

mostrar(int(df["rougeL_lora"].idxmax()))
mostrar(int(df["rougeL_lora"].idxmin()))

## 11. Demonstração interativa

Cole qualquer notícia de ciência ou tecnologia abaixo para comparar as abordagens ao vivo.

In [ ]:
# ============================================================
# DEMONSTRAÇÃO
# ============================================================

noticia_demo = """
Cole aqui o texto de uma notícia de ciência ou tecnologia.
"""

for ab in ABORDAGENS:
    texto, dt = resumir(noticia_demo, ab)
    display(Markdown(f"**{ab}** ({dt:.1f}s): {texto}"))

## 12. Recursos computacionais, limitações e autoria dos dados

**Recursos computacionais** (preencher com os valores obtidos):
- GPU: NVIDIA T4 (16 GB), Google Colab
- Modelo: Qwen2.5-1.5B-Instruct quantizado em 4 bits (NF4)
- Parâmetros treináveis: *(ver saída de `print_trainable_parameters`)*
- Tempo de treinamento: *(ver célula de treino)*

**Dados e autoria:**
- RecognaSumm — Paiola, P. H. et al. *RecognaSumm: A Novel Brazilian Summarization Dataset*. PROPOR 2024. Licença MIT.
- Os sumários de referência vêm do próprio dataset; nenhum dado foi gerado por modelo.

**Limitações** (pontos a discutir):
- Notícias truncadas em `MAX_PALAVRAS_NOTICIA` palavras: informações do final do texto não são vistas pelo modelo.
- Treino com apenas 1 época e um subconjunto pequeno (`N_TREINO`) do domínio.
- ROUGE mede sobreposição de palavras: um resumo correto com outras palavras é penalizado.
- O estilo dos sumários do dataset (ver exemplos) determina o que o modelo ajustado aprende a imitar.
- Avaliação manual feita por um único avaliador, em uma amostra pequena.
- O tempo de geração depende da GPU e da quantização; o one-shot é mais lento porque o prompt é maior.